In [3]:
pip install python-telegram-bot


  Using cached python_telegram_bot-21.10-py3-none-any.whl.metadata (17 kB)
Using cached python_telegram_bot-21.10-py3-none-any.whl (669 kB)
Note: you may need to restart the kernel to use updated packages.


In [1]:
pip install ultralytics


  Using cached ultralytics-8.3.72-py3-none-any.whl.metadata (35 kB)
  Using cached torch-2.6.0-cp312-cp312-win_amd64.whl.metadata (28 kB)
  Using cached torchvision-0.21.0-cp312-cp312-win_amd64.whl.metadata (6.3 kB)
  Using cached ultralytics_thop-2.0.14-py3-none-any.whl.metadata (9.4 kB)
  Using cached sympy-1.13.1-py3-none-any.whl.metadata (12 kB)
Using cached ultralytics-8.3.72-py3-none-any.whl (914 kB)
Using cached torch-2.6.0-cp312-cp312-win_amd64.whl (204.1 MB)
Using cached sympy-1.13.1-py3-none-any.whl (6.2 MB)
Using cached torchvision-0.21.0-cp312-cp312-win_amd64.whl (1.6 MB)
Using cached ultralytics_thop-2.0.14-py3-none-any.whl (26 kB)
Note: you may need to restart the kernel to use updated packages.


In [2]:
import nest_asyncio
import asyncio
from telegram import Update, Bot
from telegram.ext import Application, MessageHandler, filters, CallbackContext
from ultralytics import YOLO
import cv2
import numpy as np
import time
from keras.models import load_model
from collections import Counter

# التوكن الخاص بالبوت
TOKEN = '7869587173:AAFumCLEwgKBIvfULtV5yJbgi8qfOzIupWk'  # ضع هنا التوكن الخاص بك
chat_id = '672355156'  # ضع هنا معرف المستخدم أو مجموعة الدردشة التي تريد إرسال الرسائل لها

# إعداد بوت تيليغرام
bot = Bot(token=TOKEN)

# تحميل نموذج YOLO المدرب مسبقًا
yolo_model = YOLO('yolov8n.pt')

# تحميل نموذج التعرف على التعبيرات
emotion_model = load_model('model_file.h5')  # ضع هنا مسار النموذج المدرب على التعابير

# تحميل كاشف الوجه
faceDetect = cv2.CascadeClassifier('haarcascade_frontalface_default.xml')

# إعداد الفيديو
video_path = "goodservice2.mp4"
cap = cv2.VideoCapture(video_path)

# إعداد كتابة الفيديو الناتج
output_path = "output_video.mp4"
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
fps = int(cap.get(cv2.CAP_PROP_FPS))
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

# تعريف المنطقة المحددة
area_x1, area_y1 = 100, 100  # الزاوية العليا اليسرى للمنطقة
area_x2, area_y2 = 400, 400  # الزاوية السفلى اليمنى للمنطقة

# تتبع الوقت
start_time = None
person_inside_area = False

# معجم لتخزين التسميات الخاصة بالتعبيرات
labels_dict = {0: 'Angry', 1: 'Disgust', 2: 'Fear', 3: 'Happy', 4: 'Neutral', 5: 'Sad', 6: 'Surprise'}

# دالة لإرسال الرسائل إلى البوت
async def send_telegram_message(message):
    await bot.send_message(chat_id=chat_id, text=message)

# دالة لإرسال رسالة عند دخول أو خروج شخص من المنطقة
async def handle_person_in_area(person_inside):
    global person_inside_area, start_time  # استخدام global هنا
    if person_inside:
        if not person_inside_area:
            start_time = time.time()  # بدء المؤقت
            message = "شخص دخل المنطقة! بدء المؤقت."
            print(message)
            await send_telegram_message(message)
        person_inside_area = True
    else:
        if person_inside_area:
            end_time = time.time()
            duration = end_time - start_time
            message = f"الشخص غادر المنطقة. الوقت المستغرق: {duration:.2f} ثانية."
            print(message)
            await send_telegram_message(message)
        person_inside_area = False

# دالة لتحليل الفيديو وإرسال التقارير
async def process_video_and_send_notifications():
    global person_inside_area, start_time
    detected_emotions = []  # لتخزين العواطف المكتشفة
    
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        # تشغيل الكشف عن الأشخاص باستخدام YOLO
        results = yolo_model(frame)

        # رسم المنطقة المحددة
        cv2.rectangle(frame, (area_x1, area_y1), (area_x2, area_y2), (0, 255, 0), 2)

        # التحقق من وجود شخص داخل المنطقة
        found_person_in_area = False
        for result in results[0].boxes:
            if int(result.cls[0]) == 0:  # رقم التصنيف للأشخاص في COCO dataset هو 0
                x1, y1, x2, y2 = map(int, result.xyxy[0])  # إحداثيات المربع
                cv2.rectangle(frame, (x1, y1), (x2, y2), (255, 0, 0), 2)
                cv2.putText(frame, "Person", (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 0), 2)

                # التحقق من وجود الشخص داخل المنطقة بناءً على التداخل بين المربعات
                if (x1 < area_x2 and x2 > area_x1 and y1 < area_y2 and y2 > area_y1):
                    found_person_in_area = True

                # اكتشاف التعبير العاطفي من الوجه
                gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
                faces = faceDetect.detectMultiScale(gray, 1.3, 3)
                for (fx, fy, fw, fh) in faces:
                    face = gray[fy:fy+fh, fx:fx+fw]
                    resized = cv2.resize(face, (48, 48))
                    normalized = resized / 255.0
                    reshaped = np.reshape(normalized, (1, 48, 48, 1))
                    emotion_result = emotion_model.predict(reshaped)
                    label = np.argmax(emotion_result, axis=1)[0]
                    emotion = labels_dict[label]
                    detected_emotions.append(emotion)
                    cv2.putText(frame, emotion, (fx, fy-10), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)

        # التحكم في المؤقت وتحديث الرسائل
        await handle_person_in_area(found_person_in_area)

        # كتابة الإطار إلى الفيديو الناتج
        out.write(frame)

        cv2.imshow("YOLO Detection with Area Monitoring", frame)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    cap.release()
    out.release()
    cv2.destroyAllWindows()

    # إرسال التقرير النهائي للعواطف المكتشفة
    await send_emotion_report(detected_emotions)

# دالة لإرسال تقرير عن العواطف المكتشفة
async def send_emotion_report(detected_emotions):
    # حساب توزيع العواطف
    emotion_counts = Counter(detected_emotions)
    emotion_summary = '\n'.join([f"{emotion}: {count}" for emotion, count in emotion_counts.items()])
    
    # حساب النتيجة النهائية بناءً على العواطف المكتشفة
    total_count = len(detected_emotions)
    if total_count == 0:
        final_result = 0.0
    else:
        positive_emotions = ['Happy', 'Surprise']
        negative_emotions = ['Angry', 'Sad', 'Fear']
        
        positive_count = sum(1 for emotion in detected_emotions if emotion in positive_emotions)
        negative_count = sum(1 for emotion in detected_emotions if emotion in negative_emotions)
        neutral_count = sum(1 for emotion in detected_emotions if emotion == 'Neutral')

        positive_percentage = positive_count / total_count
        negative_percentage = negative_count / total_count
        neutral_percentage = neutral_count / total_count

        if positive_percentage > negative_percentage:
            final_result = 7 + (positive_percentage * 3)
        elif negative_percentage > positive_percentage:
            final_result = 1 + (negative_percentage * 2)
        else:
            final_result = 4 + (neutral_percentage * 3)

        final_result = max(1, min(10, final_result))

    # إرسال التقرير
    report = f"التقرير النهائي للتعبيرات:\n{emotion_summary}\n\nالنتيجة النهائية: {final_result:.1f}/10"
    await send_telegram_message(report)

# دالة لتكرار الرسائل التي يرسلها المستخدم
async def echo(update: Update, context: CallbackContext) -> None:
    await update.message.reply_text(f'قلت: {update.message.text}')

# دالة لتشغيل التطبيق
async def main():
    # إنشاء التطبيق (بوت Telegram)
    application = Application.builder().token(TOKEN).build()

    # إضافة المعالج لتكرار الرسائل
    application.add_handler(MessageHandler(filters.TEXT & ~filters.COMMAND, echo))

    # بدء الفيديو و معالجة الإشعارات
    await process_video_and_send_notifications()

    # بدء البوت
    await application.run_polling()

# تشغيل التطبيق
if __name__ == "__main__":
    nest_asyncio.apply()  # يسمح بتشغيل الـ event loop في بيئات مثل Jupyter أو Colab
    asyncio.run(main())  # بدء تشغيل البوت


ModuleNotFoundError: No module named 'keras'